# 🚀 Prática Hands-On: Ingestão de Dados, Classificação com LLMs e Visualização no GCP

Durante este exercício, você fará a ingestão de dados de uma API pública para o **BigQuery**, utilizará o modelo de linguagem gerenciado (**Gemini no Vertex AI**) para enriquecer esses dados com classificações automatizadas e gerará um **Gráfico de Pareto** interativo com o **Plotly**.

## 🛠️ Pré-requisitos e Setup

Vamos começar instalando e atualizando as dependências necessárias para esta prática.

In [ ]:
!pip install --upgrade google-cloud-bigquery vertexai pandas requests plotly

## Passo 0: Configuração de Variáveis Iniciais

Defina as variáveis básicas do ambiente. Substitua os valores indicados pelas suas credenciais e pelo seu identificador único.

In [ ]:
import pandas as pd
import requests
from google.cloud import bigquery

# CONFIGURAÇÕES DO PROJETO
PROJECT_ID = "seu-projeto-gcp-id" # Substitua pelo ID do seu projeto no GCP
DATASET_ID = "dataset_pratica" # Nome do dataset já criado pela equipe
NOME_SOBRENOME = "nome_sobrenome" # Ex: fulano_silva (sem espaços ou acentos)

# Nome dinâmico da sua tabela no BigQuery
TABLE_NAME = f"tb_dados_{NOME_SOBRENOME}"
TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

print(f"Sua tabela será: {TABLE_ID}")

## Passo 1 e 2: Ingestão de Dados da API Pública no BigQuery

Vamos consumir dados de uma API pública de teste (neste exemplo, comentários de usuários) e persistí-los diretamente em uma tabela individualizada no BigQuery.

In [ ]:
# 1. Buscar dados de uma API pública (Exemplo: DummyJSON API)
url = "https://dummyjson.com/comments?limit=50"
response = requests.get(url)
data = response.json()["comments"]

# Converter a resposta JSON para DataFrame Pandas
df_raw = pd.DataFrame(data)

# Selecionar apenas as colunas de interesse
df_ingest = df_raw[['id', 'body', 'likes']].copy()

# 2. Inicializar o cliente do BigQuery e criar a tabela
client = bigquery.Client(project=PROJECT_ID)

# Configuração do job de carga (sobrescreve a tabela se ela já existir)
job_config = bigquery.LoadJobConfig(
 write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Enviar os dados para o BigQuery
job = client.load_table_from_dataframe(df_ingest, TABLE_ID, job_config=job_config)
job.result() # Aguarda a conclusão da ingestão

print(f"✅ Tabela `{TABLE_ID}` criada e {len(df_ingest)} registros ingeridos com sucesso!")

## Passo 3: Conexão com LLM (Vertex AI / Gemini) para Classificação

Agora vamos consultar a tabela que acabamos de criar, conectar com o **Gemini 1.5 Flash** através da Vertex AI e classificar o sentimento/categoria de cada texto da base.

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel

# Inicializar o SDK do Vertex AI
vertexai.init(project=PROJECT_ID, location="us-central1")
model = GenerativeModel("gemini-1.5-flash")

# 1. Ler os dados diretamente da tabela criada no BigQuery
query = f"SELECT * FROM `{TABLE_ID}`"
df_bq = client.query(query).to_dataframe()

# 2. Função para classificar o texto via prompt para o Gemini
def classificar_comentario(texto):
 prompt = f"""
 Analise o texto a seguir e classifique-o em APENAS UMA das categorias abaixo:
 - Elogio
 - Crítica
 - Dúvida
 - Sugestão

 Responda EXATAMENTE apenas com a palavra da categoria escolhida.

 Texto: "{texto}"
 """
 try:
 response = model.generate_content(prompt)
 return response.text.strip()
 except Exception as e:
 return "Não Classificado"

# 3. Aplicar o modelo LLM a todos os registros da base
print("🤖 Classificando registros utilizando Gemini no Vertex AI...")
df_bq['categoria'] = df_bq['body'].apply(classificar_comentario)

# Exibir as primeiras linhas classificadas
print(df_bq[['body', 'categoria']].head())

## Passo 4: Visualização e Construção do Gráfico de Pareto (Plotly)

Com os dados devidamente categorizados pela inteligência artificial, vamos estruturar uma **Análise de Pareto** (Princípio 80/20) comparando a contagem absoluta das categorias e o percentual acumulado.

In [ ]:
import plotly.graph_objects as go

# 1. Contagem e ordenação das frequências por categoria
df_pareto = df_bq['categoria'].value_counts().reset_index()
df_pareto.columns = ['categoria', 'quantidade']
df_pareto = df_pareto.sort_values(by='quantidade', ascending=False)

# 2. Cálculo do acumulado e porcentagem acumulada
df_pareto['acumulado'] = df_pareto['quantidade'].cumsum()
df_pareto['percentual_acumulado'] = (df_pareto['acumulado'] / df_pareto['quantidade'].sum()) * 100

# 3. Construção do Gráfico de Pareto com Eixo Duplo no Plotly
fig = go.Figure()

# Eixo Y1 (Esquerda): Barras de Quantidade
fig.add_trace(go.Bar(
 x=df_pareto['categoria'],
 y=df_pareto['quantidade'],
 name='Quantidade',
 marker_color='#1f77b4'
))

# Eixo Y2 (Direita): Linha do Percentual Acumulado
fig.add_trace(go.Scatter(
 x=df_pareto['categoria'],
 y=df_pareto['percentual_acumulado'],
 name='Accumulated %',
 yaxis='y2',
 mode='lines+markers',
 line=dict(color='#d62728', width=3),
 marker=dict(size=8)
))

# Layout e Formatação do Gráfico
fig.update_layout(
 title=f'Análise de Pareto: Categorização LLM ({NOME_SOBRENOME})',
 xaxis=dict(title='Categoria Classificada'),
 yaxis=dict(title='Frequência Absoluta'),
 yaxis2=dict(
 title='Percentual Acumulado (%)',
 overlaying='y',
 side='right',
 range=[0, 110]
 ),
 template='plotly_white',
 legend=dict(x=0.65, y=1.1, orientation='h')
)

# Exibir o gráfico
fig.show()